# Biotech Healthcare Engine — Clinical Diabetes 30-Day Readmission Risk Predictor

### 1. Strategic Corporate & Clinical Context
In healthcare enterprise systems and insurance analytics, early hospital readmissions (within 30 days of discharge) incur heavy financial penalties and signal potential gaps in transitional patient care. Misclassifying an unstable, high-risk patient as safe from readmission (a False Negative) yields high re-hospitalization costs and negative quality-of-care scores. This engine processes high-dimensional clinical encounters to isolate readmission trajectories before discharge.

### 2. Technical Engineering Architecture
To maintain absolute structural parity with our enterprise microservice deployment blueprint, this healthcare diagnostic system implements:
* **Perimeter Gateways:** A strict Pydantic contract layer enforcing numerical limits and explicit clinical string tokens on incoming live patient prediction streams.
* **Feature Isolation Lines:** An immutable `ColumnTransformer` that separate continuous clinical operational counts from discrete categorical medical management markers, using safe out-of-sample imputations.
* **Imbalance Loss Optimization:** Automated class-imbalance weight calculation ($\approx 8.0 \times$ ratio) paired with a tree-boosting framework to optimize classification costs under asymmetric risk profiles.
* **Relational Audit Logging:** Instrumented pipeline telemetry tracking metrics and booster configurations inside a localized SQLite MLflow instance.

In [17]:
# =====================================================================
# CELL 1: SYSTEM INITIALIZATION & DEPENDENCY SETUP
# =====================================================================
!pip install mlflow pydantic xgboost -q

import mlflow
import pydantic
import sklearn
import pandas as pd
import numpy as np

# Bind tracking to our shared relational storage engine to preserve metadata safely
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Healthcare_Diabetes_Readmission_Architecture")

print("--- Healthcare Analytical Engine Initialized ---")
print(f"MLflow State Engine: Active (SQLite Backend)")
print(f"Pydantic Version: {pydantic.__version__}")

2026/06/16 01:49:47 INFO mlflow.tracking.fluent: Experiment with name 'Healthcare_Diabetes_Readmission_Architecture' does not exist. Creating a new experiment.


--- Healthcare Analytical Engine Initialized ---
MLflow State Engine: Active (SQLite Backend)
Pydantic Version: 2.12.3


### 3. Production Data Ingestion & Strategic Structural Cleansing
We consume the core clinical diabetes ledger ($101,766$ historical hospital records). Before divisions hit our training arrays, we drop tracking identifiers (`encounter_id`, `patient_nbr`) and sparse codes (`payer_code`) to ensure the boosting architecture relies on genuine medical signals. We normalize placeholders like `?` into explicit markers and isolate 30-day early readmissions as our primary binary label.

In [18]:
# =====================================================================
# CELL 2: DATA ACQUISITION & DATA ARCHIVE PROCESSING
# =====================================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# Ingest your uploaded clinical dataset
raw_diabetic_data = pd.read_csv('diabetic_data.csv')

print("--- Raw Patient Data Structural Audit ---")
print(f"Total Clinical Records Processed: {raw_diabetic_data.shape[0]}")
print(f"Total Raw Analytical Attributes: {raw_diabetic_data.shape[1]}")

# DATA HYGIENE: Isolate highly predictive variables and remove pure sequencing artifacts
core_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses',
    'race', 'gender', 'age', 'metformin', 'insulin', 'change', 'diabetesMed'
]

X_raw = raw_diabetic_data[core_features].copy()

# Standardize missing data tokens across clinical channels
X_raw['race'] = X_raw['race'].replace('?', 'Unknown')
X_raw['gender'] = X_raw['gender'].replace('Unknown/Invalid', 'Unknown')

# Define target: 1 = High Risk Early Readmission (<30 days), 0 = Stable / Delayed No-Readmit
y = (raw_diabetic_data['readmitted'] == '<30').astype(int)

# Partition using stratified splits to handle target class boundaries uniformly
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining Split Geometry: {X_train.shape}")
print(f"Testing Split Geometry: {X_test.shape}")
print(f"Baseline Early Readmission Prevalence: {(y_train.sum() / len(y_train)) * 100:.2f}%")

--- Raw Patient Data Structural Audit ---
Total Clinical Records Processed: 34640
Total Raw Analytical Attributes: 50

Training Split Geometry: (27712, 15)
Testing Split Geometry: (6928, 15)
Baseline Early Readmission Prevalence: 11.34%


### 4. Edge Clinical Validation Interface Contract (Pydantic Gateway)
We compile our strict **Perimeter Validation Protocol** to verify patient entry objects. Because live inference streams often arrive via public web APIs, this contract uses explicit `Field` bounds and `Literal` options to drop invalid counts or unrecognizable therapeutic string logs before they hit processing arrays.

In [19]:
# =====================================================================
# CELL 3: CLINICAL GATEWAY VALIDATION CONTRACT
# =====================================================================
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class PatientReadmissionSchema(BaseModel):
    # Enforce clear constraints verifying real clinical encounter boundaries
    time_in_hospital: int = Field(..., ge=1, le=14, description="Duration in hospital days.")
    num_lab_procedures: int = Field(..., ge=1, le=150)
    num_procedures: int = Field(..., ge=0, le=10)
    num_medications: int = Field(..., ge=1, le=100)
    number_outpatient: int = Field(..., ge=0, le=50)
    number_emergency: int = Field(..., ge=0, le=50)
    number_inpatient: int = Field(..., ge=0, le=50)
    number_diagnoses: int = Field(..., ge=1, le=20)

    race: Literal['Caucasian', 'AfricanAmerican', 'Unknown', 'Other', 'Asian', 'Hispanic']
    gender: Literal['Female', 'Male', 'Unknown']
    age: Literal['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
    metformin: Literal['No', 'Steady', 'Up', 'Down']
    insulin: Literal['No', 'Steady', 'Up', 'Down']
    change: Literal['No', 'Ch']
    diabetesMed: Literal['No', 'Yes']

    @field_validator('time_in_hospital', 'num_medications', 'number_diagnoses')
    @classmethod
    def verify_positive_metrics(cls, value: int) -> int:
        if value <= 0:
            raise ValueError("Clinical operational counts must represent true positive records.")
        return value

print("Healthcare perimeter validation contract safely compiled.")

Healthcare perimeter validation contract safely compiled.


### 5. Multi-Lane Preprocessing Isolation Pipelines
To process features uniformly without leakage during distributed testing or containerized deployments, we divide features into continuous and categorical sub-pipelines via a `ColumnTransformer`.

In [20]:
# =====================================================================
# CELL 4: MULTI-CHANNEL FEATURE ASSEMBLY GATEWAYS
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_encounter_fields = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'
]
categorical_encounter_fields = [
    'race', 'gender', 'age', 'metformin', 'insulin', 'change', 'diabetesMed'
]

numerical_clinical_route = Pipeline(steps=[
    ('median_imputer', SimpleImputer(strategy='median')),
    ('matrix_scaler', StandardScaler())
])

categorical_clinical_route = Pipeline(steps=[
    ('constant_imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

production_preprocessing_gate = ColumnTransformer(transformers=[
    ('num_track', numerical_clinical_route, numerical_encounter_fields),
    ('cat_track', categorical_clinical_route, categorical_encounter_fields)
])

print("Clinical transformation pathways completely bound.")

Clinical transformation pathways completely bound.


### 6. Tree-Boosting Optimization Loop & Clinical Audit Lineage
We couple our tissue transformation pipelines directly to an optimized tree-boosting workflow (`XGBClassifier`). Because early readmission events are highly imbalanced ($\approx 11\%$), we compute an explicit `scale_pos_weight` multiplier to balance the loss optimization natively. Everything is audited live to our MLflow relational database.

In [21]:
# =====================================================================
# CELL 5: GRADIENT BOOSTING ENGINE INITIALIZATION & DIAGNOSTIC LOGGING
# =====================================================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow.sklearn

# Capture baseline class distributions to balance target optimization thresholds natively
imbalance_ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

healthcare_readmit_pipeline = Pipeline(steps=[
    ('preprocessing_gate', production_preprocessing_gate),
    ('xgboost_engine', XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.06,
        scale_pos_weight=imbalance_ratio,
        random_state=42,
        n_jobs=-1
    ))
])

# Activate direct MLflow artifact and parameter autologging
mlflow.sklearn.autolog(log_models=True)

print("Connecting to local SQLite model registry... Running clinical optimization...")
with mlflow.start_run(run_name="Clinical_Diabetes_Readmit_Production") as active_run:

    # Train the comprehensive end-to-end pipeline
    healthcare_readmit_pipeline.fit(X_train, y_train)

    # Generate verification sets for testing metrics
    predictions = healthcare_readmit_pipeline.predict(X_test)
    probabilities = healthcare_readmit_pipeline.predict_proba(X_test)[:, 1]

    clinical_auc = roc_auc_score(y_test, probabilities)
    mlflow.log_metric("clinical_test_auc_score", clinical_auc)

    print("\n=====================================================================")
    print("--- METRICS TRACE COMPLETED & REGISTERED ---")
    print(f"Logged SQL Run ID: {active_run.info.run_id}")
    print(f"Readmission ROC-AUC Performance Metric: {clinical_auc:.4f}")
    print("=====================================================================\n")
    print(classification_report(y_test, predictions))

Connecting to local SQLite model registry... Running clinical optimization...


2026/06/16 01:51:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/16 01:51:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils


--- METRICS TRACE COMPLETED & REGISTERED ---
Logged SQL Run ID: 42a3464eb0884dde9ae013fb29d19e72
Readmission ROC-AUC Performance Metric: 0.6392

              precision    recall  f1-score   support

           0       0.92      0.64      0.76      6142
           1       0.17      0.56      0.26       786

    accuracy                           0.63      6928
   macro avg       0.54      0.60      0.51      6928
weighted avg       0.83      0.63      0.70      6928



### 7. Clinical Operational Asset Serialization
We freeze the end-to-end data transformation extraction pipeline and decision graph into a production-tier serialized pickle archive, ready for portable service deployment.

In [22]:
# =====================================================================
# CELL 6: PRODUCTION PIPELINE SERIALIZATION & DOWNLOAD PROTOCOL
# =====================================================================
from google.colab import files
import joblib

# Export operational system artifact
joblib.dump(healthcare_readmit_pipeline, 'healthcare_readmit_pipeline.pkl')

files.download('healthcare_readmit_pipeline.pkl')
print("Successfully generated and downloaded 'healthcare_readmit_pipeline.pkl'!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully generated and downloaded 'healthcare_readmit_pipeline.pkl'!
